In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math as m

from ipywidgets import interact, FloatSlider, IntSlider, Play, jslink, VBox, fixed


In [2]:
#Cambell
period = 100*60*60*24*365.25
periastron_passage_time = 0.4*period
eccentricity = 0.5
semi_major_axis = 149597870700 
inclination = 60 * np.pi/180
longitude_periastron = 250 * np.pi/180
angle_of_ascending_node = 120 * np.pi/180

num_periods = 1
mass_quotient = 1 

$x_{n+1} = x_{n} \frac{f(xn)}{f'(xn)}$

In [3]:
def trobarE(t,periastron_passage_time,eccentricity,mu):
    #Newton-Raphson
    #x_{n+1} = x_{n} \frac{f(xn)}{f'(xn)} 
    
    Eo = 0
    E = 1  
    ε = 1e-6

    M = mu*(t-periastron_passage_time)

    def Ef (M,Ei,eccentricity):
        return (Ei - eccentricity*m.sin(Ei) - M)/(1 - eccentricity*m.cos(Ei))
    
    while abs(E-Eo) > ε:
        Eo = E
        E = Eo - Ef(M,Eo,eccentricity)
        


    return E
        

In [4]:
def orbites (num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    t = 0
    dt = period*1e-3
    n = int(period * num_periods / dt)
    mu = 2*np.pi/period
    E = 0
    ε = 1e-10
    G = 6.67 * 1e-11
    cnst = 4*np.pi**2*semi_major_axis**3/(G*period**2)
    m1 = cnst/(1+mass_quotient)
    m2 = mass_quotient*m1
    x1_list = np.zeros(n)
    y1_list = np.zeros(n)
    x2_list = np.zeros(n)
    y2_list = np.zeros(n)
    xcm_list = np.zeros(n)
    ycm_list = np.zeros(n)
    t_list= np.zeros(n)
    i=0

    
    c1 = m2/(m1 + m2)
    c2 = - m1/(m1 + m2)

    if inclination == 0:
        angle_of_ascending_node = 0

    if eccentricity == 0:
         longitude_periastron = 0

    A = semi_major_axis*(np.cos(angle_of_ascending_node)*np.cos(longitude_periastron) - np.sin(angle_of_ascending_node)*np.sin(longitude_periastron)*np.cos(inclination))
    B = semi_major_axis*(np.sin(angle_of_ascending_node)*np.cos(longitude_periastron) + np.cos(angle_of_ascending_node)*np.sin(longitude_periastron)*np.cos(inclination))
    F = semi_major_axis*(-np.cos(angle_of_ascending_node)*np.sin(longitude_periastron) - np.sin(angle_of_ascending_node)*np.cos(longitude_periastron)*np.cos(inclination))
    G = semi_major_axis*(-np.sin(angle_of_ascending_node)*np.sin(longitude_periastron) + np.cos(angle_of_ascending_node)*np.cos(longitude_periastron)*np.cos(inclination))


 
    xf = A*(-eccentricity)
    yf = B*(-eccentricity)
    

                
    while t < period*num_periods:
        E = trobarE(t,periastron_passage_time,eccentricity,mu)

        X = np.cos(E) - eccentricity
        Y = np.sqrt(1-eccentricity**2) * np.sin(E)


        x = A*X + F*Y
        y = B*X + G*Y

        x1 = c1*x
        x2 = c2*x
        y1 = c1*y
        y2 = c2*y        

        xcm = x2*c1 - x1*c2
        ycm = y2*c1 - y1*c2


        xcm_list[i]=xcm
        ycm_list[i]=ycm
        x1_list[i]=x1
        y1_list[i]=y1
        x2_list[i]=x2
        y2_list[i]=y2
        t_list[i]=t

        i += 1
        t = t+dt

    return xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf

In [5]:
def plot_orbites(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf = orbites(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    xcm = xcm_list[-1]
    ycm = ycm_list[-1]
    x1 = x1_list[-1]
    y1 = y1_list[-1]
    x2 = x2_list[-1]
    y2 = y2_list[-1]

    
    plt.scatter (xcm, ycm, color = 'orange', s = 15, zorder = 3)
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter (x1_list, y1_list, c=t_list, cmap = 'pink', s = 5, zorder = 1)
    plt.plot (x1, y1, '*', color = 'yellow', markersize = 15, zorder = 3)
    plt.scatter (x2_list, y2_list, c=t_list, cmap = 'bone', s = 5, zorder = 1)
    plt.plot (x2, y2, '*', color = 'yellow', markersize = 15*mass_quotient, zorder = 3)
    plt.gca().set_box_aspect(1)
    plt.gca().set_aspect('equal')
    plt.show()


In [6]:
interact(
    plot_orbites,
    num_periods=fixed(num_periods),
    period = FloatSlider(min=periastron_passage_time, max = 3*period, value=period),
    periastron_passage_time = FloatSlider(min=0, max=period, value=periastron_passage_time),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient)
)


interactive(children=(FloatSlider(value=3155760000.0, description='period', max=9467280000.0, min=1262304000.0…

<function __main__.plot_orbites(num_periods, period, periastron_passage_time, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)>

In [7]:
xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf = orbites(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

def plot_frame(frame):
    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.show()

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=len(t_list)-1,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_frame, frame=slider)

display(ui, out)


interactive(children=(IntSlider(value=0, description='frame', max=999), Output()), _dom_classes=('widget-inter…

<function __main__.plot_frame(frame)>

In [8]:
def plot_frame_mod(frame,num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf = orbites(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.show()

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)



jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_frame_mod, frame=slider,
               num_periods=fixed(num_periods),
    period = FloatSlider(min=periastron_passage_time, max = 3*period, value=period),
    periastron_passage_time = FloatSlider(min=0, max=period, value=periastron_passage_time),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient))

display(ui, out)

interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=3155760000.0, descri…

<function __main__.plot_frame_mod(frame, num_periods, period, periastron_passage_time, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)>

In [9]:
def orbita3D(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    t = 0
    dt = period*1e-3
    n = int(period * num_periods / dt)
    mu = 2*np.pi/period
    E = 0
    ε = 1e-10
    G = 6.67 * 1e-11
    cnst = 4*np.pi**2*semi_major_axis**3/(G*period**2)
    m1 = cnst/(1+mass_quotient)
    m2 = m1*mass_quotient
    zcm_list = np.zeros(n)
    z1_list = np.zeros(n)
    z2_list = np.zeros(n)
    i=0
    
    
    c1 = m2/(m1 + m2)
    c2 = - m1/(m1 + m2)
    
    
    
    xcm_list, ycm_list, x1_list, y1_list, x2_list, y2_list, t_list, xf, yf = orbites(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    if inclination == 0:
        angle_of_ascending_node = 0

    if eccentricity == 0:
         longitude_periastron = 0

    H = semi_major_axis*np.sin(longitude_periastron)*np.sin(inclination)
    G = semi_major_axis*np.cos(longitude_periastron)*np.sin(inclination)
    
    zf = H*(-eccentricity)
    
            
    while t < period*num_periods:
        E = trobarE(t,periastron_passage_time,eccentricity,mu)

        X = np.cos(E) - eccentricity
        Y = np.sqrt(1-eccentricity**2) * np.sin(E)
        
        z = H*X + G*Y

        z1 = c1*z
        z2 = c2*z 
        zcm = z2*c1 - z1*c2

        zcm_list[i]=zcm
        z1_list[i]=z1
        z2_list[i]=z2

        i += 1
        t = t+dt

    
    return xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf

In [10]:
def plot_orbites3D(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):

    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)
    
    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list, y1_list, z1_list, c=t_list, cmap='pink', s=5)
    sc2 = ax.scatter(x2_list, y2_list, z2_list, c=t_list, cmap='bone', s=5)
    
    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[-1], ycm_list[-1], zcm_list[-1], '*', color='orange', s=100)
    ax.scatter(x1_list[-1], y1_list[-1], z1_list[-1], '*', color='yellow', s=100)
    ax.scatter(x2_list[-1], y2_list[-1], z2_list[-1], '*', color='yellow', s=100*mass_quotient)
    
    ax.set_box_aspect([1,1,1])
    plt.show()


    xcm = xcm_list[-1]
    ycm = ycm_list[-1]
    x1 = x1_list[-1]
    y1 = y1_list[-1]
    x2 = x2_list[-1]
    y2 = y2_list[-1]

    
    plt.scatter (xcm, ycm, color = 'orange', s = 15, zorder = 3)
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter (x1_list, y1_list, c=t_list, cmap = 'pink', s = 5, zorder = 1)
    plt.plot (x1, y1, '*', color = 'yellow', markersize = 15, zorder = 3)
    plt.scatter (x2_list, y2_list, c=t_list, cmap = 'bone', s = 5, zorder = 1)
    plt.plot (x2, y2, '*', color = 'yellow', markersize = 15*mass_quotient, zorder = 3)
    plt.gca().set_box_aspect(1)
    plt.gca().set_aspect('equal')
    plt.show()


In [11]:
interact(
    plot_orbites3D,
    num_periods=fixed(num_periods),
    period = period,
    periastron_passage_time = FloatSlider(min=0, max=period, value=periastron_passage_time),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient)
)


interactive(children=(FloatSlider(value=3155760000.0, description='period', max=9467280000.0, min=-3155760000.…

<function __main__.plot_orbites3D(num_periods, period, periastron_passage_time, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)>

In [12]:
xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

def plot_frame3D(frame):
    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)
    

    ax.set_box_aspect([1,1,1])
    plt.show()

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.show()

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=len(t_list)-1,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_frame3D, frame=slider)

display(ui, out)


interactive(children=(IntSlider(value=0, description='frame', max=999), Output()), _dom_classes=('widget-inter…

<function __main__.plot_frame3D(frame)>

In [13]:
def plot_frame_mod3D(frame,num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient):
    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)
    
    ax.set_box_aspect([1,1,1])
    plt.show()

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.scatter (xf, yf, color = 'blue', s = 15, zorder = 3)
    plt.scatter(xcm_list[frame], ycm_list[frame], color='orange', s=15)
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15*mass_quotient)

    plt.gca().set_aspect('equal')
    plt.gca().set_box_aspect(1)
    plt.show()

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)



jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_frame_mod3D, frame=slider,
               num_periods=fixed(num_periods),
    period = FloatSlider(min=periastron_passage_time, max = 3*period, value=period),
    periastron_passage_time = FloatSlider(min=0, max=period, value=periastron_passage_time),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = FloatSlider(min=0, max=np.pi, value=inclination),
    longitude_periastron = FloatSlider(min=0, max=2*np.pi, value=longitude_periastron),
    angle_of_ascending_node = FloatSlider(min=0, max=2*np.pi, value=angle_of_ascending_node),
    mass_quotient = FloatSlider(min=0, max=1, value=mass_quotient))

display(ui, out)

interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=3155760000.0, descri…

<function __main__.plot_frame_mod3D(frame, num_periods, period, periastron_passage_time, eccentricity, semi_major_axis, inclination, longitude_periastron, angle_of_ascending_node, mass_quotient)>

In [14]:
def plot_orbites3Dpolars(phi, theta, num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,mass_quotient):

    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,0,0,0,mass_quotient)
    
    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list, y1_list, z1_list, c=t_list, cmap='pink', s=5)
    sc2 = ax.scatter(x2_list, y2_list, z2_list, c=t_list, cmap='bone', s=5)
    
    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[-1], ycm_list[-1], zcm_list[-1], '*', color='orange', s=100)
    ax.scatter(x1_list[-1], y1_list[-1], z1_list[-1], '*', color='yellow', s=100)
    ax.scatter(x2_list[-1], y2_list[-1], z2_list[-1], '*', color='yellow', s=100*mass_quotient)

    ax.view_init(elev=np.degrees(theta), azim=np.degrees(phi))

    ax.set_box_aspect([1,1,1])
    plt.show()
    
phi_slider = FloatSlider(min=0, max=2*np.pi, step=0.01, value=0, description='phi')
theta_slider = FloatSlider(min=0, max=np.pi, step=0.01, value=np.pi/4, description='theta')

interact(
    plot_orbites3Dpolars, phi=phi_slider, theta=theta_slider,
    num_periods=fixed(num_periods),
    period = period,
    periastron_passage_time = FloatSlider(min=0, max=period, value=periastron_passage_time),
    eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
    semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
    inclination = fixed(inclination),
    longitude_periastron = fixed(longitude_periastron),
    angle_of_ascending_node = fixed(angle_of_ascending_node),
    mass_quotient = fixed(mass_quotient)
)


interactive(children=(FloatSlider(value=0.0, description='phi', max=6.283185307179586, step=0.01), FloatSlider…

<function __main__.plot_orbites3Dpolars(phi, theta, num_periods, period, periastron_passage_time, eccentricity, semi_major_axis, mass_quotient)>

In [15]:
xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,inclination,longitude_periastron,angle_of_ascending_node,mass_quotient)

def plot_frame3Dpolars(frame,phi,theta):
    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')
    
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)

    ax.view_init(elev=np.degrees(theta), azim=np.degrees(phi))

    ax.set_box_aspect([1,1,1])
    plt.show()

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)

phi_slider = FloatSlider(min=0, max=2*np.pi, step=0.01, value=0, description='phi')
theta_slider = FloatSlider(min=0, max=np.pi, step=0.01, value=np.pi/4, description='theta')

slider = IntSlider(
    min=0,
    max=len(t_list)-1,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_frame3Dpolars, frame=slider, phi=phi_slider, theta=theta_slider)

display(ui, out)


interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=0.0, description='ph…

<function __main__.plot_frame3Dpolars(frame, phi, theta)>

In [16]:
def plot_frame_mod3Dpolars(frame,phi,theta,num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,mass_quotient):
    xcm_list, ycm_list, zcm_list, x1_list, y1_list, z1_list, x2_list, y2_list, z2_list, t_list, xf, yf, zf = orbita3D(num_periods,period,periastron_passage_time,eccentricity,semi_major_axis,0,0,0,mass_quotient)
    fig = plt.figure(figsize=(6,6))
    
    ax = fig.add_subplot(111, projection='3d')
    sc1 = ax.scatter(x1_list[:frame], y1_list[:frame], z1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    sc2 = ax.scatter(x2_list[:frame], y2_list[:frame], z2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    
    ax.scatter(xf, yf, zf, color='blue', s=100)
    ax.scatter(xcm_list[frame], ycm_list[frame], zcm_list[frame], color='orange', s=100)
    ax.scatter(x1_list[frame], y1_list[frame], z1_list[frame], '*', color='yellow', s=100)
    ax.scatter(x2_list[frame], y2_list[frame], z2_list[frame], '*', color='yellow', s=100*mass_quotient)
    
    ax.view_init(elev=np.degrees(theta), azim=np.degrees(phi))
    ax.set_box_aspect([1,1,1])
    
    plt.show()
    
    # widgets de reproducció
    
play = Play( value=0, min=0, max=len(t_list)-1, step=1, interval=30 )
phi_slider = FloatSlider(min=0, max=2*np.pi, step=0.01, value=0, description='phi')
theta_slider = FloatSlider(min=0, max=np.pi, step=0.01, value=np.pi/4, description='theta')
jslink((play, 'value'), (slider, 'value'))
ui = VBox([play, slider])

out = interact(plot_frame_mod3Dpolars, frame=slider, phi=phi_slider, theta=theta_slider,
               num_periods=fixed(num_periods),
               period = FloatSlider(min=periastron_passage_time, max = 3*period, value=period),
               periastron_passage_time = FloatSlider(min=0, max=period, value=periastron_passage_time),
               eccentricity = FloatSlider(min=0, max=0.999999999, step=0.1, value=eccentricity),
               semi_major_axis = FloatSlider(min=0, max=semi_major_axis*3, value=semi_major_axis),
               inclination = fixed(inclination),
               longitude_periastron = fixed(longitude_periastron),
               angle_of_ascending_node = fixed(angle_of_ascending_node),
               mass_quotient = fixed(mass_quotient))

display(ui, out)

interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=0.0, description='ph…

<function __main__.plot_frame_mod3Dpolars(frame, phi, theta, num_periods, period, periastron_passage_time, eccentricity, semi_major_axis, mass_quotient)>